<a href="https://colab.research.google.com/github/shreejaywaghayee/Tic-Tac-Toe-Game-Design-Using-Python/blob/main/Tic_Tac_Toe_Game_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random

import ipywidgets as widgets
from IPython.display import HTML, display

WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]

COORD_NAMES = ["A1", "A2", "A3", "B1", "B2", "B3", "C1", "C2", "C3"]


# ---------- Game logic ----------
def find_line(board):
    """Return (mark, line) if three matching marks align, otherwise None."""
    for a, b, c in WIN_LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a], (a, b, c)
    return None


def evaluate_outcome(board, misere=False):
    """
    Return (winner_mark, line) or None.
    In Misère mode, the player who makes 3-in-a-row LOSES, so the other player wins.
    """
    found = find_line(board)
    if not found:
        return None
    mark, line = found
    winner = other(mark) if misere else mark
    return winner, line


def empty_squares(board):
    return [i for i, mark in enumerate(board) if mark == " "]


def other(mark):
    return "O" if mark == "X" else "X"


def minimax(board, turn, ai, depth=0, alpha=-100, beta=100, misere=False):
    """Alpha-beta minimax supporting both standard and Misère rules."""
    outcome = evaluate_outcome(board, misere=misere)
    if outcome:
        winner, _ = outcome
        return (10 - depth) if winner == ai else (depth - 10)
    avail = empty_squares(board)
    if not avail:
        return 0

    if turn == ai:
        best = -100
        for sq in avail:
            board[sq] = turn
            val = minimax(board, other(turn), ai, depth + 1, alpha, beta, misere)
            board[sq] = " "
            best = max(best, val)
            alpha = max(alpha, best)
            if beta <= alpha:
                break
        return best
    else:
        best = 100
        for sq in avail:
            board[sq] = turn
            val = minimax(board, other(turn), ai, depth + 1, alpha, beta, misere)
            board[sq] = " "
            best = min(best, val)
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best


def best_move(board, ai, misere=False):
    avail = empty_squares(board)
    if not avail:
        return None
    if len(avail) == 9:
        # In Misère, corners/edges are safer openings than center
        return random.choice([0, 2, 6, 8] if misere else [0, 2, 4, 6, 8])

    best_score, best_squares = None, []
    for sq in avail:
        board[sq] = ai
        score = minimax(board, other(ai), ai, depth=1, misere=misere)
        board[sq] = " "
        if best_score is None or score > best_score:
            best_score, best_squares = score, [sq]
        elif score == best_score:
            best_squares.append(sq)
    return random.choice(best_squares)


# ---------- Styling (Dark & Light Themes) ----------
STYLE = """
<style>
  .ttt-wrap {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    padding: 18px 20px;
    border-radius: 18px;
    max-width: 390px;
    transition: background 0.25s ease, color 0.25s ease;
  }
  /* Dark Theme (Default) */
  .ttt-dark {
    background: linear-gradient(145deg, #0F172A, #1E293B);
    color: #F8FAFC;
    box-shadow: 0 12px 30px rgba(0, 0, 0, 0.35);
    border: 1px solid #334155;
  }
  .ttt-dark .ttt-sub { color: #94A3B8; }
  .ttt-dark .ttt-score {
    background: #1E293B;
    border: 1px solid #334155;
  }
  .ttt-dark .ttt-score small { color: #94A3B8; }
  .ttt-dark .ttt-score b { color: #F8FAFC; }
  .ttt-dark .ttt-cell {
    background: #1E293B !important;
    border: 1px solid #334155 !important;
  }
  .ttt-dark .ttt-cell:hover {
    background: #293548 !important;
    border-color: #475569 !important;
  }
  .ttt-dark .ttt-history { color: #94A3B8; }

  /* Light Theme */
  .ttt-light {
    background: #FFFFFF;
    color: #0F172A;
    box-shadow: 0 10px 25px rgba(15, 23, 42, 0.08);
    border: 1px solid #E2E8F0;
  }
  .ttt-light .ttt-sub { color: #64748B; }
  .ttt-light .ttt-score {
    background: #F8FAFC;
    border: 1px solid #E2E8F0;
  }
  .ttt-light .ttt-score small { color: #64748B; }
  .ttt-light .ttt-score b { color: #0F172A; }
  .ttt-light .ttt-cell {
    background: #F1F5F9 !important;
    border: 1px solid #E2E8F0 !important;
  }
  .ttt-light .ttt-cell:hover {
    background: #E2E8F0 !important;
  }
  .ttt-light .ttt-history { color: #64748B; }

  /* Shared Elements */
  .ttt-title { font-size: 24px; font-weight: 800; letter-spacing: -0.5px; margin: 0; }
  .ttt-sub   { font-size: 12.5px; margin: 2px 0 10px 0; }

  .ttt-cell {
    font-size: 42px !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    box-shadow: none !important;
    transition: transform 0.08s ease, background 0.15s ease !important;
  }
  .ttt-x    { color: #10B981 !important; }
  .ttt-o    { color: #8B5CF6 !important; }
  .ttt-win  {
    background: rgba(16, 185, 129, 0.22) !important;
    border: 2px solid #10B981 !important;
  }
  .ttt-lose-line {
    background: rgba(244, 63, 94, 0.22) !important;
    border: 2px solid #F43F5E !important;
  }
  .ttt-hint {
    background: rgba(245, 158, 11, 0.22) !important;
    border: 2px dashed #F59E0B !important;
  }

  .ttt-score {
    display: inline-block;
    width: 102px;
    margin: 4px 3px;
    padding: 8px 0;
    text-align: center;
    border-radius: 12px;
  }
  .ttt-score small { font-weight: 700; font-size: 11px; letter-spacing: 0.5px; }
  .ttt-score b     { display: block; font-size: 22px; margin-top: 2px; }

  .ttt-status { font-weight: 700; font-size: 15px; margin: 10px 0 4px 0; min-height: 22px; }
  .ttt-history { font-size: 11.5px; min-height: 16px; margin-bottom: 6px; font-family: monospace; }
</style>
"""


# ---------- Game UI & Controller ----------
def start():
    state = {
        "board": [" "] * 9,
        "history": [],          # list of (square, mark)
        "turn": "X",
        "human_mark": "X",
        "over": False,
        "mode": "hard",         # two | easy | medium | hard | misere
        "theme": "dark",
        "scores": {"X": 0, "Draws": 0, "O": 0},
    }

    display(HTML(STYLE))

    header = widgets.HTML()
    mode_picker = widgets.Dropdown(
        options=[
            ("Mode: Unbeatable AI", "hard"),
            ("Mode: Medium AI", "medium"),
            ("Mode: Easy AI", "easy"),
            ("Mode: Misère (3-in-a-row Loses!)", "misere"),
            ("Mode: 2 Players (Local)", "two"),
        ],
        value="hard",
        layout=widgets.Layout(width="215px"),
    )
    side_picker = widgets.ToggleButtons(
        options=[("Play X", "X"), ("Play O", "O")],
        value="X",
        style={"button_width": "58px"},
    )
    theme_btn = widgets.Button(
        description="☀️ Light",
        layout=widgets.Layout(width="82px"),
    )

    scoreboard = widgets.HTML()
    status = widgets.HTML()
    history_bar = widgets.HTML()

    cells = [
        widgets.Button(description=" ", layout=widgets.Layout(width="104px", height="104px"))
        for _ in range(9)
    ]
    for cell in cells:
        cell.add_class("ttt-cell")

    grid = widgets.GridBox(
        cells,
        layout=widgets.Layout(
            grid_template_columns="repeat(3, 104px)",
            grid_gap="8px",
            margin="6px 0 10px 0",
        ),
    )

    new_btn = widgets.Button(description="New Game", button_style="success", layout=widgets.Layout(width="104px"))
    undo_btn = widgets.Button(description="↩ Undo", layout=widgets.Layout(width="84px"))
    hint_btn = widgets.Button(description="💡 Hint", layout=widgets.Layout(width="84px"))
    reset_btn = widgets.Button(description="Reset", layout=widgets.Layout(width="70px"))

    container = widgets.VBox(
        [
            header,
            widgets.HBox([mode_picker, theme_btn], layout=widgets.Layout(justify_content="space-between")),
            side_picker,
            scoreboard,
            status,
            history_bar,
            grid,
            widgets.HBox([new_btn, undo_btn, hint_btn, reset_btn]),
        ]
    )
    container.add_class("ttt-wrap")
    container.add_class("ttt-dark")

    def is_misere():
        return state["mode"] == "misere"

    def ai_mark():
        return other(state["human_mark"])

    def update_header():
        subtitle = (
            "Avoid making 3 in a row — force the AI to connect!"
            if is_misere()
            else "Connect three in a row to win"
        )
        header.value = (
            f"<div class='ttt-title'>Tic Tac Toe</div>"
            f"<div class='ttt-sub'>{subtitle}</div>"
        )

    def clear_hints():
        for c in cells:
            c.remove_class("ttt-hint")

    def render_board():
        for i, cell in enumerate(cells):
            mark = state["board"][i]
            cell.description = mark
            cell.remove_class("ttt-x")
            cell.remove_class("ttt-o")
            cell.remove_class("ttt-win")
            cell.remove_class("ttt-lose-line")
            cell.remove_class("ttt-hint")
            if mark == "X":
                cell.add_class("ttt-x")
            elif mark == "O":
                cell.add_class("ttt-o")

        outcome = evaluate_outcome(state["board"], misere=is_misere())
        if outcome:
            _, line = outcome
            css_class = "ttt-lose-line" if is_misere() else "ttt-win"
            for idx in line:
                cells[idx].add_class(css_class)

    def show_scores():
        s = state["scores"]
        x_label = "X (YOU)" if state["mode"] != "two" and state["human_mark"] == "X" else "X"
        o_label = "O (YOU)" if state["mode"] != "two" and state["human_mark"] == "O" else "O"
        scoreboard.value = "".join(
            f"<div class='ttt-score'><small>{label}</small><b>{s[key]}</b></div>"
            for label, key in [(x_label, "X"), ("DRAWS", "Draws"), (o_label, "O")]
        )

    def show_status(text, colour="inherit"):
        status.value = f"<div class='ttt-status' style='color:{colour}'>{text}</div>"

    def show_history():
        if not state["history"]:
            history_bar.value = "<div class='ttt-history'>Moves: none yet</div>"
        else:
            moves_str = " → ".join(f"{m}:{COORD_NAMES[sq]}" for sq, m in state["history"])
            history_bar.value = f"<div class='ttt-history'>Moves: {moves_str}</div>"

    def choose_ai_square():
        avail = empty_squares(state["board"])
        if not avail:
            return None
        mode = state["mode"]
        if mode == "easy":
            return random.choice(avail)
        if mode == "medium":
            return best_move(state["board"], ai_mark()) if random.random() < 0.5 else random.choice(avail)
        if mode == "misere":
            return best_move(state["board"], ai_mark(), misere=True)
        return best_move(state["board"], ai_mark(), misere=False)

    def apply_move(square):
        clear_hints()
        mark = state["turn"]
        state["board"][square] = mark
        state["history"].append((square, mark))
        render_board()
        show_history()

        outcome = evaluate_outcome(state["board"], misere=is_misere())
        if outcome:
            winner, _ = outcome
            state["over"] = True
            state["scores"][winner] += 1
            colour = "#10B981" if winner == "X" else "#8B5CF6"
            if is_misere():
                loser = other(winner)
                show_status(f"{loser} made 3-in-a-row — {winner} wins!", colour)
            else:
                show_status(f"{winner} wins!", colour)
        elif not empty_squares(state["board"]):
            state["over"] = True
            state["scores"]["Draws"] += 1
            show_status("It's a draw!", "#94A3B8")
        else:
            state["turn"] = other(mark)
            if state["mode"] == "two":
                show_status(f"{state['turn']}'s turn")
            else:
                your_turn = state["turn"] == state["human_mark"]
                show_status(f"Your turn ({state['human_mark']})" if your_turn else "AI's turn...")
        show_scores()

    def new_game(_=None):
        state.update(board=[" "] * 9, history=[], turn="X", over=False)
        side_picker.layout.display = "none" if state["mode"] == "two" else "flex"
        update_header()
        render_board()
        show_scores()
        show_history()

        if state["mode"] == "two":
            show_status("X's turn")
        elif state["human_mark"] == "X":
            show_status("Your turn (X)")
        else:
            show_status("AI opens as X...")
            sq = choose_ai_square()
            if sq is not None:
                apply_move(sq)

    def reset_scores(_=None):
        state["scores"] = {"X": 0, "Draws": 0, "O": 0}
        show_scores()

    def undo_move(_=None):
        if not state["history"]:
            return
        # Undo 2 plies in vs-AI mode so it returns to the human's turn, or 1 ply in 2-player
        steps = 2 if (state["mode"] != "two" and len(state["history"]) >= 2 and not (state["over"] and state["turn"] == ai_mark())) else 1
        for _ in range(steps):
            if state["history"]:
                sq, mark = state["history"].pop()
                state["board"][sq] = " "
                state["turn"] = mark
        state["over"] = False
        render_board()
        show_history()
        if state["mode"] == "two":
            show_status(f"{state['turn']}'s turn")
        else:
            show_status(f"Your turn ({state['human_mark']})")

    def show_hint(_=None):
        if state["over"]:
            return
        clear_hints()
        sq = best_move(state["board"], state["turn"], misere=is_misere())
        if sq is not None:
            cells[sq].add_class("ttt-hint")
            show_status(f"Hint: try {COORD_NAMES[sq]}", "#F59E0B")

    def toggle_theme(_=None):
        if state["theme"] == "dark":
            state["theme"] = "light"
            container.remove_class("ttt-dark")
            container.add_class("ttt-light")
            theme_btn.description = "🌙 Dark"
        else:
            state["theme"] = "dark"
            container.remove_class("ttt-light")
            container.add_class("ttt-dark")
            theme_btn.description = "☀️ Light"

    def change_mode(change):
        state["mode"] = change["new"]
        new_game()

    def change_side(change):
        state["human_mark"] = change["new"]
        new_game()

    def click(square):
        if state["over"] or state["board"][square] != " ":
            return
        if state["mode"] != "two" and state["turn"] != state["human_mark"]:
            return
        apply_move(square)
        if not state["over"] and state["mode"] != "two" and state["turn"] == ai_mark():
            sq = choose_ai_square()
            if sq is not None:
                apply_move(sq)

    for i, cell in enumerate(cells):
        cell.on_click(lambda _btn, n=i: click(n))
    new_btn.on_click(new_game)
    reset_btn.on_click(reset_scores)
    undo_btn.on_click(undo_move)
    hint_btn.on_click(show_hint)
    theme_btn.on_click(toggle_theme)
    mode_picker.observe(change_mode, names="value")
    side_picker.observe(change_side, names="value")

    new_game()
    display(container)


start()